# 04 — Validação, consolidação e série espectral representativa

Este caderno recebe os CSVs exportados pelos cinco painéis. Ele verifica a estrutura e confere cada identificador contra os candidatos publicados. Um avaliador principal cobre as cinco classes; a avaliação independente das classes 02 e 09 é preservada como controle de reprodutibilidade. Em caso de discordância, prevalece a avaliação principal.

Quando há mais de um candidato validado como vegetação fotossinteticamente ativa em uma data, o espectro representativo é o candidato com maior escore PPI. O desempate é determinístico pela ordem sistemática. A seleção não transforma `EM01`, `EM02` etc. em indivíduos persistentes: esses rótulos são posições relativas dentro de cada data.

## Preparação e contrato do CSV

Cada arquivo precisa conter exatamente as cinco colunas abaixo, na mesma ordem. Candidatos múltiplos são separados por `|`; a escolha `NENHUM` exige `candidate_ids` vazio.

In [ ]:
from pathlib import Path
import json
import hashlib

import numpy as np
import pandas as pd
import yaml


def localizar_repositorio(inicio=None):
    """Localiza a raiz sem pressupor o sistema operacional ou caminho local."""
    atual = Path(inicio or Path.cwd()).resolve()
    for candidato in (atual, *atual.parents):
        if (candidato / "config" / "study.yaml").is_file() and (candidato / "data").is_dir():
            return candidato
    raise FileNotFoundError("A raiz do repositório não foi encontrada.")


RAIZ = localizar_repositorio()
with (RAIZ / "config" / "study.yaml").open(encoding="utf-8") as arquivo:
    CONFIG = yaml.safe_load(arquivo)

CLASSES = {str(codigo).zfill(2): nome for codigo, nome in CONFIG["classes"].items()}
BANDAS = list(CONFIG["bands"])
SAIDA = RAIZ / "outputs" / "standalone"
SAIDA.mkdir(parents=True, exist_ok=True)

print(f"Repositório: {RAIZ}")
print(f"Classes: {', '.join(CLASSES)}")
print(f"Bandas: {', '.join(BANDAS)}")

### Catálogo de candidatos válidos

A tabela de referência reúne os candidatos das cinco classes e normaliza códigos e datas. Esse catálogo será usado para confirmar que cada decisão exportada pelo painel aponta para um candidato real da classe e da data informadas.

In [ ]:
COLUNAS_REVISAO = [
    "class_code",
    "date",
    "active_vegetation_endmembers",
    "candidate_ids",
    "reviewed_at",
]

candidatos_referencia = pd.concat(
    [
        pd.read_parquet(RAIZ / "data" / "candidates" / f"candidates_class_{codigo}.parquet")
        for codigo in CLASSES
    ],
    ignore_index=True,
)
candidatos_referencia["class_code"] = candidatos_referencia["class_code"].astype(str).str.zfill(2)
candidatos_referencia["date"] = pd.to_datetime(candidatos_referencia["date"]).dt.strftime("%Y-%m-%d")
candidatos_por_id = candidatos_referencia.set_index("candidate_id", drop=False).to_dict("index")

print(f"Candidatos de referência: {len(candidatos_referencia):,}")

## Validação de um arquivo

Para cada linha, o procedimento testa:

1. nomes e ordem das colunas;
2. unicidade de `classe × data`;
3. compatibilidade entre rótulos e identificadores;
4. existência do candidato;
5. correspondência de classe, data e rótulo;
6. cobertura de todas as datas que possuem candidatos no painel da classe.

A validação falha antes de gravar qualquer produto consolidado se uma dessas condições não for satisfeita.

In [ ]:
def validar_revisao(caminho, exigir_cobertura=True):
    caminho = Path(caminho)
    revisoes = pd.read_csv(caminho, dtype=str).fillna("")
    erros = []

    if list(revisoes.columns) != COLUNAS_REVISAO:
        erros.append(f"Colunas inválidas; esperado: {COLUNAS_REVISAO}")
        return revisoes, erros

    revisoes["class_code"] = revisoes["class_code"].astype(str).str.zfill(2)
    revisoes["date"] = pd.to_datetime(revisoes["date"], errors="coerce").dt.strftime("%Y-%m-%d")
    if revisoes["date"].isna().any():
        erros.append("Existem datas inválidas.")
    if revisoes.duplicated(["class_code", "date"]).any():
        erros.append("Existem combinações classe × data duplicadas.")

    for linha in revisoes.itertuples(index=False):
        chave = f"{linha.class_code}/{linha.date}"
        if linha.active_vegetation_endmembers == "NENHUM":
            if linha.candidate_ids:
                erros.append(f"NENHUM deve ter candidate_ids vazio: {chave}")
            continue

        rotulos = [valor for valor in linha.active_vegetation_endmembers.split("|") if valor]
        ids = [valor for valor in linha.candidate_ids.split("|") if valor]
        if not rotulos or len(rotulos) != len(ids):
            erros.append(f"Seleção incompatível com identificadores: {chave}")
            continue
        if len(rotulos) != len(set(rotulos)) or len(ids) != len(set(ids)):
            erros.append(f"Candidatos repetidos: {chave}")

        for rotulo, candidate_id in zip(rotulos, ids):
            candidato = candidatos_por_id.get(candidate_id)
            if candidato is None:
                erros.append(f"Identificador inexistente: {candidate_id}")
                continue
            if (
                candidato["class_code"] != linha.class_code
                or candidato["date"] != linha.date
                or candidato["endmember_label"] != rotulo
            ):
                erros.append(f"Identificador incompatível com {rotulo}: {chave}")

    if exigir_cobertura:
        for codigo in sorted(revisoes["class_code"].dropna().unique()):
            datas_esperadas = set(
                candidatos_referencia.loc[candidatos_referencia["class_code"].eq(codigo), "date"]
            )
            datas_recebidas = set(revisoes.loc[revisoes["class_code"].eq(codigo), "date"])
            faltantes = datas_esperadas - datas_recebidas
            extras = datas_recebidas - datas_esperadas
            if faltantes:
                erros.append(f"Classe {codigo}: faltam {len(faltantes)} datas disponíveis no painel.")
            if extras:
                erros.append(f"Classe {codigo}: existem {len(extras)} datas sem candidatos.")

    return revisoes, erros

## Descoberta e consolidação dos arquivos recebidos

Os avaliadores devem salvar os CSVs em `reviews/incoming/class_XX/`. Todos os arquivos são validados individualmente. A consolidação científica utiliza os cinco arquivos principais; os dois arquivos da avaliação independente não são descartados nem fundidos automaticamente e permanecem disponíveis para o cálculo da concordância.

In [ ]:
ARQUIVOS_PRINCIPAIS = {
    "01": "reviews/incoming/class_01/wtss1000_active_vegetation_class_01.csv",
    "02": "reviews/incoming/class_02/wtss1000_active_vegetation_class_02.csv",
    "09": "reviews/incoming/class_09/wtss1000_active_vegetation_class_09.csv",
    "10": "reviews/incoming/class_10/wtss1000_active_vegetation_class_10.csv",
    "11": "reviews/incoming/class_11/wtss1000_active_vegetation_class_11.csv",
}
ARQUIVOS_SECUNDARIOS = {
}

arquivos_revisao = [RAIZ / caminho for caminho in [*ARQUIVOS_PRINCIPAIS.values(), *ARQUIVOS_SECUNDARIOS.values()]]
tabelas_por_arquivo = {}
relatorio_revisoes = []

for caminho in arquivos_revisao:
    tabela, erros = validar_revisao(caminho, exigir_cobertura=True)
    relatorio_revisoes.append(
        {
            "file": caminho.relative_to(RAIZ).as_posix(),
            "rows": len(tabela),
            "errors": len(erros),
            "status": "PASS" if not erros else "FAIL",
        }
    )
    if erros:
        raise ValueError(f"{caminho.name}: " + "; ".join(erros[:20]))
    tabela["review_source"] = caminho.relative_to(RAIZ).as_posix()
    tabelas_por_arquivo[caminho.relative_to(RAIZ).as_posix()] = tabela

revisoes_principais = []
for codigo, relativo in ARQUIVOS_PRINCIPAIS.items():
    tabela = tabelas_por_arquivo[relativo].copy()
    tabela["reviewer_id"] = "rodrigo"
    tabela["decision_rule"] = "PRIMARY_REVIEW"
    if codigo in ARQUIVOS_SECUNDARIOS:
        secundaria = tabelas_por_arquivo[ARQUIVOS_SECUNDARIOS[codigo]].set_index("date")
        tabela["decision_rule"] = [
            if linha.active_vegetation_endmembers == secundaria.loc[linha.date, "active_vegetation_endmembers"]
            and linha.candidate_ids == secundaria.loc[linha.date, "candidate_ids"]
            else "PRIMARY_REVIEW_PRECEDENCE_ON_DISAGREEMENT"
            for linha in tabela.itertuples(index=False)
        ]
    revisoes_principais.append(tabela)

revisoes_consolidadas = pd.concat(revisoes_principais, ignore_index=True)
revisoes_consolidadas = revisoes_consolidadas.sort_values(["class_code", "date"]).reset_index(drop=True)
assert not revisoes_consolidadas.duplicated(["class_code", "date"]).any()
assert len(revisoes_consolidadas) == 918

destino = RAIZ / "reviews" / "validated"
destino.mkdir(parents=True, exist_ok=True)
revisoes_consolidadas.to_parquet(destino / "reviews_consolidadas.parquet", index=False)
revisoes_consolidadas.to_csv(destino / "reviews_consolidadas.csv", index=False, encoding="utf-8")
print(f"Revisões principais consolidadas: {len(revisoes_consolidadas):,} decisões.")

pd.DataFrame(relatorio_revisoes)

## Construção da série representativa

Seja \(V_{c,t}\) o conjunto de candidatos validados como vegetação fotossinteticamente ativa na classe \(c\) e data \(t\). O espectro representativo é

\[
e^*_{c,t}=\arg\max_{e\in V_{c,t}}
[\operatorname{PPI}(e),-\operatorname{ordem}(e)].
\]

Primeiro é maximizado o escore PPI. Se houver empate, vence o menor valor de `systematic_order`. Quando o avaliador registra `NENHUM`, a data permanece ausente. Para cada banda \(b\), a série resultante é

\[
Y_{c,b,t}=\rho_b(e^*_{c,t}).
\]

O código abaixo só produz a série quando existem revisões consolidadas.

In [ ]:
def construir_serie_representativa(revisoes, candidatos, bandas):
    linhas = []
    por_id = candidatos.set_index("candidate_id", drop=False)

    for revisao in revisoes.itertuples(index=False):
        if revisao.active_vegetation_endmembers == "NENHUM":
            continue
        ids = [valor for valor in revisao.candidate_ids.split("|") if valor]
        selecionados = por_id.loc[ids].copy()
        if isinstance(selecionados, pd.Series):
            selecionados = selecionados.to_frame().T
        selecionados = selecionados.reset_index(drop=True)
        selecionados = selecionados.sort_values(
            ["ppi_score", "systematic_order", "candidate_id"], ascending=[False, True, True]
        )
        escolhido = selecionados.iloc[0]
        registro = {
            "class_code": revisao.class_code,
            "date": revisao.date,
            "candidate_id": escolhido["candidate_id"],
            "endmember_label": escolhido["endmember_label"],
            "ppi_score": int(escolhido["ppi_score"]),
            "validated_candidates": "|".join(ids),
        }
        registro.update({banda: float(escolhido[banda]) for banda in bandas})
        linhas.append(registro)
    return pd.DataFrame(linhas)


if revisoes_consolidadas.empty:
    serie_representativa = pd.DataFrame()
    print("Série não construída: as revisões ainda não foram recebidas.")
else:
    serie_representativa = construir_serie_representativa(
        revisoes_consolidadas, candidatos_referencia, BANDAS
    )
    destino = RAIZ / "reviews" / "validated"
    serie_representativa.to_parquet(destino / "serie_espectral_representativa.parquet", index=False)
    serie_representativa.to_csv(destino / "serie_espectral_representativa.csv", index=False, encoding="utf-8")
    display(serie_representativa.groupby("class_code").size().rename("valid_dates"))

## Próxima etapa científica

A decomposição temporal deve ser iniciada somente depois que cada classe alcançar ao menos 80% das datas validadas. Cada uma das dez bandas será decomposta separadamente por STL robusto com periodicidade de 23 compostos. Datas marcadas como `NENHUM` continuarão ausentes nos produtos observados; qualquer interpolação necessária ao STL deverá ser registrada em uma série auxiliar e não substituir os valores observados.